In [ ]:
!git clone https://github.com/AliaksandrSiarohin/first-order-model

In [ ]:
%cd first-order-model

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
import imageio
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from skimage.transform import resize
from IPython.display import HTML
import warnings
warnings.filterwarnings("ignore")

In [ ]:
#read video
read_video=imageio.get_reader('/content/gdrive/MyDrive/deepfake/first-order-model/00.mp4')

#Create an empty list i.e driving video for video frame.
driving_video=[]

#for all frames in vide, append frames to the list i,e driving_video
for i in read_video:
  driving_video.append(i)

#extract video frame rate from meta data
fps = read_video.get_meta_data()['fps']

#close the video
read_video.close()

print(f"The video frame rate is: {fps}")

#read and resize source image retaining 3 channels
read_image=imageio.imread('/content/gdrive/MyDrive/deepfake/first-order-model/zubeen.png')
read_image=resize(read_image,(256,256))
read_image=read_image[..., :3]

#resizig driving_video retaining 3 channels for all elements i.e frame from the above driving_video list
driving_video=[resize(f,(256,256))[...,:3] for f in driving_video]


In [ ]:
#create a function to make the frames from video ready to create the plot
def display(source,driving,generated=None):
  fig=plt.figure(figsize=(7+4*(generated is not None),6)) #width, height=6
  imf = [] #Initialize an empty list to store frames for animation
  for i in range (len(driving)):
    temp=[source]#create a temp list to hold images for current frame of animation
    temp.append(driving[i])#add current frame of driving_video to this temp list
    if generated is not None:
      temp.append(generated[i]) #If generated is provided, the corresponding frame from the generated video (generated[i]) is also added to temp.
      im=plt.imshow(np.concatenate(temp,axis=1),animation=True)#Join the images in temp side by side along the horizontal axis.
      plt.axis('off') #Hides the axes, ticks, and labels
      imf.append(im)#Appends the current frame (im) to the empty list of animation

  ani=animation.ArtistAnimation(fig,imf,interval=50,repeat_delay=1000)#Create animation using ArtistAnimation() from animation library
  plt.close() #close the plot
  return ani #eturn animation object
HTML(display(read_image,driving_video).to_html5_video())#Display the Animation in HTML5 Video Format







In [ ]:
#old code
def display(source, driving, generated=None):
    fig = plt.figure(figsize=(8 + 4 * (generated is not None), 6))
    ims = []
    for i in range(len(driving)):
        cols = [source]
        cols.append(driving[i])
        if generated is not None:
            cols.append(generated[i])
        im = plt.imshow(np.concatenate(cols, axis=1), animated=True)
        plt.axis('off')
        ims.append([im])
    ani=animation.ArtistAnimation(fig,ims,interval=50,repeat_delay=1000)
    plt.close()
    return ani

HTML(display(read_image, driving_video).to_html5_video())

In [ ]:
pip install -r /content/first-order-model/requirements.txt

In [ ]:
import sys
sys.path.append('/content/first-order-model/sync_batchnorm')

In [ ]:
pip install ffmpeg

In [ ]:
from demo import load_checkpoints

generator, kp_detector = load_checkpoints(config_path='/content/first-order-model/config/vox-256.yaml',
                                          checkpoint_path='/content/gdrive/MyDrive/deepfake/first-order-model/vox-cpk.pth.tar')

In [ ]:

from demo import make_animation
from skimage import img_as_ubyte

predictions = make_animation(read_image, driving_video, generator, kp_detector, relative=True) # Relative Keypoint Displacement

# Save resulting video
imageio.mimsave('/content/gdrive/MyDrive/deepfake/gen.mp4', [img_as_ubyte(frame) for frame in predictions], fps=fps)
# Video can be downloaded from /content folder

HTML(display(read_image, driving_video, predictions).to_html5_video())


In [ ]:
# Import libraries
import imageio
import numpy as np
import torch
from skimage.transform import resize
from torchvision import transforms

# Define file paths
driving_video_path = '/content/gdrive/MyDrive/deepfake/first-order-model/00.mp4'
source_image_path = '/content/gdrive/MyDrive/deepfake/first-order-model/zubeen.png'
gen_video_path = '/content/gdrive/MyDrive/deepfake/gen.mp4'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Load video frames and resize them
def load_video_frames(video_path):
    reader = imageio.get_reader('/content/drive/MyDrive/deepfake/gen.mp4')
    video_frames = [resize(frame, (256, 256))[..., :3] for frame in reader]
    reader.close()
    return video_frames

# Load generated video frames
generated_frames = load_video_frames('/content/drive/MyDrive/deepfake/gen.mp4')
print(f"Number of frames in generated video: {len(generated_frames)}")

In [ ]:
# Define a preprocessing function for the frames
def preprocess_frames(frames):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return torch.stack([transform(frame) for frame in frames])

# Preprocess the frames
processed_frames = preprocess_frames(generated_frames)
print(f"Shape of processed frames: {processed_frames.shape}")

In [ ]:
# Example: Load a pre-trained deepfake detection model (placeholder)
class DeepFakeDetector(torch.nn.Module):
    def __init__(self, pretrained=True):
        super(DeepFakeDetector, self).__init__()
        # Define a simple model (replace with your actual implementation)
        self.fc = torch.nn.Linear(256*256*3, 1)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return torch.sigmoid(self.fc(x))

# Initialize the detector
detector = DeepFakeDetector(pretrained=True)
detector.eval()

print("DeepFakeDetector initialized.")

In [ ]:
import torch

# Assume processed_frames is your list of frames
# Ensure processed_frames is a list of tensors with proper preprocessing

# Perform deepfake detection on each frame
predictions = []  # List to store predictions
threshold = 0.5  # Set a threshold for classification (adjust as needed)

with torch.no_grad():  # Disable gradient calculation for inference
    for frame in processed_frames:
        # Step 1: Ensure the frame has the correct data type
        frame = frame.float()  # Convert to float if necessary

        # Step 2: Add batch dimension
        frame = frame.unsqueeze(0)  # Add batch dimension to shape (1, C, H, W)

        # Step 3: Normalize the frame (optional, based on model requirements)
        # Example normalization: divide pixel values by 255 to scale to [0, 1]
        frame = frame / 255.0

        # Step 4: Pass the frame through the model
        output = detector(frame)  # Output should be a tensor

        # Step 5: Handle the output correctly
        # Check if output is scalar, tensor, or has multiple values
        if output.numel() == 1:  # Single value (scalar)
            predictions.append(output.item())  # Convert scalar to Python float
        else:
            # If output is a tensor with multiple values, handle accordingly
            predictions.append(output[0].item())  # Adjust based on output shape

# Classify frames based on a threshold
labels = ['Fake' if pred >= threshold else 'Real' for pred in predictions]

# Display results
fake_count = labels.count('Fake')
real_count = labels.count('Real')

print(f"Total Frames: {len(labels)}")
print(f"Fake Frames: {fake_count}")
print(f"Real Frames: {real_count}")

In [ ]:
import cv2
from skimage import img_as_ubyte

# Annotate video frames
annotated_frames = []
for frame, label in zip(generated_frames, labels):
    annotated_frame = (frame * 255).astype(np.uint8)
    label_text = f"Label: {label}"
    annotated_frame = cv2.putText(
        annotated_frame, label_text, (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2, cv2.LINE_AA
    )
    annotated_frames.append(annotated_frame)

# Save the annotated video
annotated_video_path = '/content/drive/MyDrive/deepfake/gen.mp4'
imageio.mimsave(annotated_video_path, [img_as_ubyte(frame) for frame in annotated_frames], fps=30)
print(f"Annotated video saved to: {annotated_video_path}")